# Swin-UMamba trên BTXRD: các thí nghiệm thay đổi siêu tham số (Google Colab)

Notebook này dùng **đúng pipeline** của `run_colab_a100 (1).ipynb` (nnU-Net v2, cấu hình 2d, mặt nạ 3 lớp: 0 = nền, 1 = u lành tính, 2 = u ác tính). Kiến trúc và trainer giữ nguyên; mỗi thí nghiệm chỉ thay **một nhóm siêu tham số** so với cấu hình cơ sở B0.

**Vì sao không chạy cấu hình gốc của bài báo?** Cấu hình gốc là 1000 epoch (dừng sớm ở 350), 250 iteration/epoch, đóng băng encoder 10 epoch. Với khoảng 218 s/epoch trên GPU L4, 350 epoch mất khoảng 21 giờ cho **một** lần chạy, vượt giới hạn một phiên Colab. Vì vậy cấu hình 20 epoch đã báo cáo được dùng làm **mốc so sánh (B0)**.

Do giới hạn thời gian, notebook chỉ chạy **một thí nghiệm huấn luyện (E1, khoảng 1,5 giờ trên L4)**. **Số epoch (20), số iteration/epoch (200), batch (8) và kích thước ảnh (512) giữ nguyên như B0**, nên tổng số patch được học (20 × 200 × 8 = 32.000) và thời gian huấn luyện tương đương B0. Các siêu tham số còn lại được đổi **cùng lúc**:

| Siêu tham số | B0 | E1 | Lý do thay đổi |
|---|---|---|---|
| LR ban đầu (`INITIAL_LR`) | 3e-4 | **5e-4** | Chỉ có 20 epoch (bài báo cần ~350); LR lớn hơn giúp hội tụ nhanh hơn trong ngân sách ngắn, cosine vẫn giảm về 1e-6 ở cuối |
| Đóng băng encoder (`FREEZE_EPOCHS`) | 2 | **1** | Encoder pretrained được fine-tune thêm 1 epoch |
| Trọng số CE (nền, lành, ác) | 1, 1, 2 | **1, 2, 3** | Hạ tương đối trọng số nền để giảm số ca bỏ sót (mặt nạ rỗng), vẫn ưu tiên lớp ác hiếm |
| Oversample vùng u (`OVERSAMPLE_FG`) | 0,66 | **0,8** | Thêm patch chứa u (tác dụng nhỏ nếu patch gần bằng cả ảnh) |

| Mã | Nội dung | Thời gian (L4) |
|---|---|---|
| **B0** | Cấu hình cơ sở 20 epoch; đã train, notebook chỉ đánh giá lại và đo thời gian | ~10–15 phút |
| **E1** | Cấu hình đã chỉnh ở bảng trên | ~1 giờ 30 (train ~75 phút + cài đặt, tiền xử lý, đánh giá) |

**Lưu ý khi diễn giải:** vì nhiều siêu tham số được đổi cùng lúc, E1 chỉ trả lời câu hỏi *"bộ siêu tham số mới có tốt hơn B0 không?"*. Nó **không** cho biết tham số nào gây ra khác biệt. Với một lần chạy duy nhất, chênh lệch nhỏ (khoảng ±0,01–0,02 Dice) có thể chỉ do ngẫu nhiên.

**Cách chạy**
1. Chuẩn bị Kaggle token và Google Drive giống notebook gốc (xem README).
2. Ở ô **0. Cấu hình**, đặt `EXP = "B0"` rồi *Runtime → Run all*. Lần đầu notebook tự khởi động lại kernel sau khi cài thư viện, nên cần bấm *Run all* thêm một lần.
3. Đặt `EXP = "E1"` rồi *Run all*. Mỗi thí nghiệm có thư mục kết quả riêng `MyDrive/Swin-UMamba-BTXRD/results/<run_name>/`, nên các lần chạy không ghi đè lên nhau. Nếu Colab bị ngắt, chỉ cần *Run all* lại: notebook train tiếp từ `checkpoint_latest`. Thí nghiệm đã train xong thì được bỏ qua bước train.
4. Sau khi chạy xong các thí nghiệm, mục **15** gộp mọi `summary_run.json` thành bảng so sánh `results/comparison.csv` và biểu đồ `results/tradeoff.png`.

## 0. Cấu hình thí nghiệm

In [ ]:
import os, sys, glob, json, shutil, subprocess

# ================= CHỌN THÍ NGHIỆM =================
EXP = "E1"          # <-- "B0" (chỉ đánh giá lại lần chạy cơ sở) hoặc "E1" (thí nghiệm ~1,5 giờ)

# Cấu hình cơ sở B0 = lần chạy 20 epoch trong báo cáo (KHÔNG sửa, để còn so sánh được)
BASE = dict(EPOCHS=20, ITERS_PER_EPOCH=200, BATCH_SIZE=8, INITIAL_LR=3e-4, FREEZE_EPOCHS=2,
            OVERSAMPLE_FG=0.66, CE_CLASS_WEIGHTS="1,1,2", MAX_SIZE=512)

# E1 giữ nguyên EPOCHS / ITERS_PER_EPOCH / BATCH_SIZE / MAX_SIZE (=> cùng số patch, cùng thời gian ~ B0),
# chỉ đổi các siêu tham số còn lại (khoá viết HOA ghi đè BASE)
EXPERIMENTS = {
    "B0": dict(run_name="ep20",     desc="Cấu hình cơ sở (20 epoch)"),
    "E1": dict(run_name="E1_tuned", desc="LR 5e-4, freeze 1, CE (1,2,3), oversample 0,8",
               INITIAL_LR=5e-4, FREEZE_EPOCHS=1, CE_CLASS_WEIGHTS="1,2,3", OVERSAMPLE_FG=0.8),
}
assert EXP in EXPERIMENTS, f"EXP phải là một trong {list(EXPERIMENTS)}"
CFG = {**BASE, **{k: v for k, v in EXPERIMENTS[EXP].items() if k.isupper()}}

EPOCHS           = CFG["EPOCHS"]
ITERS_PER_EPOCH  = CFG["ITERS_PER_EPOCH"]
BATCH_SIZE       = CFG["BATCH_SIZE"]
INITIAL_LR       = CFG["INITIAL_LR"]          # cosine giảm về 1e-6 sau EPOCHS epoch
OVERSAMPLE_FG    = CFG["OVERSAMPLE_FG"]       # tỉ lệ patch bắt buộc chứa khối u (nnU-Net gốc: 0.33)
CE_CLASS_WEIGHTS = CFG["CE_CLASS_WEIGHTS"]    # trọng số CE cho (nền, lành, ác). "" = không trọng số
MAX_SIZE         = CFG["MAX_SIZE"]            # resize cạnh dài nhất về <= MAX_SIZE

FREEZE_EPOCHS    = CFG["FREEZE_EPOCHS"]       # số epoch đầu đóng băng encoder pretrained (tối thiểu 1)

TRAINER    = "nnUNetTrainerSwinUMambaColab"   # Swin-UMamba + pretrain ImageNet (giữ nguyên ở mọi thí nghiệm)
FOLD       = "0"          # fold 0 = đúng split train/val có sẵn của BTXRD
VAL_ITERS  = 20           # số batch dùng tính pseudo-Dice sau mỗi epoch
SAVE_EVERY = 1            # chỉ ghi đè checkpoint_latest (mỗi epoch)

# ================= DỮ LIỆU =================
KAGGLE_DATA_SLUG = "nguyetanhmai/btxrd-split"
KAGGLE_ENV_SLUG  = "nguyetanhmai/setup-envi"
LABEL_MODE   = "benign_malignant"   # 0 = nền, 1 = lành tính, 2 = ác tính
# Mỗi độ phân giải là một dataset nnU-Net riêng (dữ liệu tiền xử lý khác nhau)
DATASET_ID   = {512: 705, 768: 706, 1024: 707}[MAX_SIZE]
DATASET_NAME = "Dataset705_BTXRD" if MAX_SIZE == 512 else f"Dataset{DATASET_ID:03d}_BTXRD{MAX_SIZE}"

# ================= ĐƯỜNG DẪN =================
REPO_URL  = "https://github.com/67htb5vn/Swin-UMamba.git"
BRANCH    = "main"
REPO      = "/content/Swin-UMamba"
TMP       = "/content/nnunet"            # raw + preprocessed: đĩa local của Colab (nhanh, mất khi ngắt runtime)
KAGGLE_DL = "/content/kaggle_data"
USE_DRIVE  = True
DRIVE_ROOT = "/content/drive/MyDrive/Swin-UMamba-BTXRD"
RESULTS_ROOT = f"{DRIVE_ROOT if USE_DRIVE else '/content'}/results"
RUN_NAME     = EXPERIMENTS[EXP]["run_name"]
RESULTS_DIR  = f"{RESULTS_ROOT}/{RUN_NAME}"      # data/nnUNet_results của repo sẽ trỏ tới đây
WHEEL_STORE  = f"{DRIVE_ROOT if USE_DRIVE else '/content'}/wheels"

PRETRAIN_URL = "https://github.com/MzeroMiko/VMamba/releases/download/%23v0cls/vssmtiny_dp01_ckpt_epoch_292.pth"

os.environ.update({
    "TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD": "1",              # torch>=2.6 mặc định weights_only=True -> không load được checkpoint nnU-Net
    "nnUNet_n_proc_DA": str(min(12, os.cpu_count() or 4)),
    "nnUNet_compile": "f",
    "MPLBACKEND": "Agg",
    "SUM_EPOCHS": str(EPOCHS),
    "SUM_ITERS": str(ITERS_PER_EPOCH),
    "SUM_SAVE_EVERY": str(SAVE_EVERY),
    "SUM_VAL_ITERS": str(VAL_ITERS),
    "SUM_FREEZE_EPOCHS": str(FREEZE_EPOCHS),
    "SUM_LR": str(INITIAL_LR),
    "SUM_OVERSAMPLE": str(OVERSAMPLE_FG),
    "SUM_CE_WEIGHTS": CE_CLASS_WEIGHTS,
})

for k in ["EPOCHS", "ITERS_PER_EPOCH", "BATCH_SIZE", "MAX_SIZE"]:   # điều kiện để so sánh công bằng với B0
    assert CFG[k] == BASE[k], f"{k} phải giữ nguyên như B0 ({BASE[k]})"
print(f"[{EXP}] {EXPERIMENTS[EXP]['desc']}")
diff = {k: (BASE[k], CFG[k]) for k in BASE if BASE[k] != CFG[k]}
print("Khác B0:", ", ".join(f"{k}: {a!r} -> {b!r}" for k, (a, b) in diff.items()) or "(không)")
print(f"EPOCHS={EPOCHS} | FREEZE={FREEZE_EPOCHS} | BATCH={BATCH_SIZE} | ITERS={ITERS_PER_EPOCH} | LR={INITIAL_LR} "
      f"| OVERSAMPLE={OVERSAMPLE_FG} | CE={CE_CLASS_WEIGHTS or 'none'} | MAX_SIZE={MAX_SIZE}")
print("Dataset:", DATASET_NAME, "| Kết quả lưu tại:", RESULTS_DIR)

## 1. Google Drive và kiểm tra GPU

In [ ]:
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    os.makedirs(DRIVE_ROOT, exist_ok=True)

!nvidia-smi
import torch
assert torch.cuda.is_available(), "Runtime chưa bật GPU: Runtime -> Change runtime type -> GPU"
gpu = torch.cuda.get_device_name(0)
mem = torch.cuda.get_device_properties(0).total_memory / 2**30
print(f"GPU: {gpu} | {mem:.0f} GB | CPU: {os.cpu_count()} | torch {torch.__version__} | CUDA {torch.version.cuda}")
if mem < 30 and BATCH_SIZE > 8:
    print(f"CẢNH BÁO: GPU chỉ có {mem:.0f} GB -> nên giảm BATCH_SIZE xuống 8 (hoặc 4) rồi chạy lại ô cấu hình.")

## 2. Tải dataset từ Kaggle
Thứ tự tìm thông tin đăng nhập Kaggle: Colab Secrets (`KAGGLE_USERNAME`/`KAGGLE_KEY`, hoặc `KAGGLE_API_TOKEN`) → `~/.kaggle/kaggle.json` → `MyDrive/Swin-UMamba-BTXRD/kaggle.json` → `MyDrive/kaggle.json` → upload thủ công.

In [ ]:
!pip install -q --upgrade kaggle

def setup_kaggle_auth():
    try:
        from google.colab import userdata
        for k in ["KAGGLE_USERNAME", "KAGGLE_KEY", "KAGGLE_API_TOKEN"]:
            try:
                v = userdata.get(k)
                if v:
                    os.environ[k] = v.strip()   # bỏ khoảng trắng/xuống dòng thừa khi dán
            except Exception:
                pass
    except ImportError:
        pass
    if (os.environ.get("KAGGLE_USERNAME") and os.environ.get("KAGGLE_KEY")) or os.environ.get("KAGGLE_API_TOKEN"):
        return "Colab Secrets"
    kj = os.path.expanduser("~/.kaggle/kaggle.json")
    os.makedirs(os.path.dirname(kj), exist_ok=True)
    for p in [kj, f"{DRIVE_ROOT}/kaggle.json", "/content/drive/MyDrive/kaggle.json"]:
        if os.path.isfile(p):
            if p != kj:
                shutil.copy(p, kj)
            os.chmod(kj, 0o600)
            return p
    from google.colab import files
    print("Không tìm thấy thông tin đăng nhập Kaggle -> hãy upload file kaggle.json")
    up = files.upload()
    name = next(n for n in up if n.endswith(".json"))
    open(kj, "wb").write(up[name])
    os.chmod(kj, 0o600)
    return "upload"

def kaggle_download(slug, dest):
    if os.path.isdir(dest) and any(os.scandir(dest)):
        print("Đã có sẵn:", dest)
        return
    os.makedirs(dest, exist_ok=True)
    print("Đang tải", slug, "->", dest)
    # Cách 1: kaggle CLI (in rõ thông báo lỗi nếu thất bại)
    r = subprocess.run(["kaggle", "datasets", "download", "-d", slug, "-p", dest, "--unzip"],
                       capture_output=True, text=True)
    print(r.stdout[-2000:], r.stderr[-2000:])
    if r.returncode == 0 and any(os.scandir(dest)):
        return
    # Cách 2: kagglehub (dùng cùng thông tin đăng nhập)
    print("kaggle CLI lỗi -> thử kagglehub...")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--upgrade", "kagglehub"], check=False)
    import kagglehub
    path = kagglehub.dataset_download(slug)
    print("kagglehub tải về:", path)
    shutil.rmtree(dest)
    os.symlink(path, dest)

def check_kaggle_auth():
    r = subprocess.run(["kaggle", "datasets", "list", "--mine"], capture_output=True, text=True)
    print("Kiểm tra đăng nhập Kaggle (datasets của tài khoản):")
    print(r.stdout[-1500:] or r.stderr[-1500:])
    for k in ["KAGGLE_USERNAME", "KAGGLE_KEY", "KAGGLE_API_TOKEN"]:
        v = os.environ.get(k, "")
        print(f"  {k}: {'(trống)' if not v else ('độ dài ' + str(len(v)) + (', CÓ khoảng trắng/xuống dòng thừa!' if v != v.strip() else ''))}")

print("Kaggle auth:", setup_kaggle_auth())
check_kaggle_auth()
kaggle_download(KAGGLE_DATA_SLUG, f"{KAGGLE_DL}/btxrd")
kaggle_download(KAGGLE_ENV_SLUG, f"{KAGGLE_DL}/env")

def find_btxrd(root):
    """Tìm thư mục gốc chứa train/ val/ test/ (mỗi tập gồm images/ + Annotations/)."""
    for p in sorted(glob.glob(f"{root}/**/train/Annotations", recursive=True)):
        d = os.path.dirname(os.path.dirname(p))
        if all(os.path.isdir(os.path.join(d, s, "images")) for s in ["train", "val", "test"]):
            return d
    return None

BTXRD_DIR = find_btxrd(f"{KAGGLE_DL}/btxrd")
assert BTXRD_DIR, f"Không tìm thấy cấu trúc train/val/test trong {KAGGLE_DL}/btxrd"
print("BTXRD_DIR =", BTXRD_DIR, "->", sorted(os.listdir(BTXRD_DIR)))
for s in ["train", "val", "test"]:
    print(f"  {s}: {len(os.listdir(f'{BTXRD_DIR}/{s}/images'))} ảnh, {len(glob.glob(f'{BTXRD_DIR}/{s}/Annotations/*.json'))} json")
print("Wheel từ Kaggle:", glob.glob(f"{KAGGLE_DL}/env/**/*.whl", recursive=True))

## 3. Clone repo từ GitHub

In [ ]:
if not os.path.isdir(REPO):
    subprocess.run(["git", "clone", "--depth", "1", "-b", BRANCH, REPO_URL, REPO], check=True)
else:
    subprocess.run(["git", "-C", REPO, "pull"], check=False)
!git -C {REPO} log --oneline -1

## 4. Cài môi trường
- Cài nnU-Net bản sửa đổi và các thư viện thuần Python (mất khoảng 1–2 phút).
- `mamba-ssm` và `causal-conv1d`: notebook thử lần lượt các bộ wheel theo thứ tự (1) wheel Colab đã lưu trên Drive, (2) wheel trong dataset Kaggle `setup-envi`. Mỗi bộ được kiểm tra bằng cách chạy thử kernel CUDA.
- Wheel `setup-envi` được build trên Kaggle. Nếu phiên bản torch/CUDA/Python của Colab khác Kaggle thì wheel đó không chạy được, và notebook sẽ **tự build lại** (lần đầu mất khoảng 10–30 phút). Wheel mới được lưu lên Drive (`MyDrive/Swin-UMamba-BTXRD/wheels/<env_tag>/`), nên các lần sau chỉ cần cài lại.

In [ ]:
# Cài torch 2.11 + nnU-Net + numpy 2 rồi tự restart 1 lần (trước khi kernel nạp numpy)
MARK = "/content/.env_ready"
if not os.path.exists(MARK):
    !pip install -q torch==2.11.0 torchvision --index-url https://download.pytorch.org/whl/cu130
    !cd {REPO}/swin_umamba && pip install -q -e . && pip install -q "monai==1.4.0" timm torchinfo einops numba packaging ninja
    !pip install -q "numpy>=2.1,<2.6"
    open(MARK, "w").close()
    os.kill(os.getpid(), 9)   # restart -> bấm Run all lần nữa
import torch, numpy; print("torch", torch.__version__, "| numpy", numpy.__version__)

In [ ]:
import torch
ENV_TAG = f"torch{torch.__version__.split('+')[0]}-cu{torch.version.cuda}-py{sys.version_info.major}{sys.version_info.minor}"
print("ENV_TAG hiện tại:", ENV_TAG)

def wheel_candidates():
    """Các thư mục chứa đủ 2 wheel; ưu tiên thư mục có env_tag khớp, rồi đến wheel trên Drive."""
    dirs = sorted({os.path.dirname(p) for root in [WHEEL_STORE, f"{KAGGLE_DL}/env"]
                   for p in glob.glob(f"{root}/**/mamba_ssm-*.whl", recursive=True)})
    cands = []
    for d in dirs:
        if not glob.glob(f"{d}/causal_conv1d-*.whl"):
            continue
        tag_file = f"{d}/env_tag.txt"
        tag = open(tag_file).read().strip() if os.path.isfile(tag_file) else "?"
        print(f"  tìm thấy wheel: {d}  (env_tag={tag})")
        cands.append((tag != ENV_TAG, not d.startswith(WHEEL_STORE), d))
    return [d for *_, d in sorted(cands)]

def mamba_ok():
    test = ("import torch; from mamba_ssm.ops.selective_scan_interface import selective_scan_fn; "
            "u=torch.randn(1,4,16,device='cuda'); A=-torch.rand(4,8,device='cuda'); B=torch.randn(1,8,16,device='cuda'); "
            "selective_scan_fn(u,u,A,B,B); import causal_conv1d")
    return subprocess.run([sys.executable, "-c", test], capture_output=True).returncode == 0

%cd {REPO}/swin_umamba
!pip install -q --no-deps -e .
# setup.py ghim monai==1.3.0 (không tương thích numpy 2) -> dùng 1.4.0, API các block giữ nguyên
!pip install -q "monai==1.4.0" timm torchinfo einops numba packaging ninja

ok = False
for d in wheel_candidates():
    print("Thử cài mamba từ:", d)
    !pip install -q --no-deps --force-reinstall {d}/causal_conv1d-*.whl {d}/mamba_ssm-*.whl
    ok = mamba_ok()
    print("  -> dùng được" if ok else "  -> KHÔNG tương thích với torch hiện tại")
    if ok:
        break

if not ok:
    print("Không có wheel phù hợp -> build wheel cho môi trường Colab (lần đầu sẽ lâu)...")
    os.environ["MAX_JOBS"] = str(os.cpu_count() or 4)
    WHEEL_OUT = f"{WHEEL_STORE}/{ENV_TAG}"
    !rm -rf "{WHEEL_OUT}" && mkdir -p "{WHEEL_OUT}"
    !pip wheel -q --no-build-isolation --no-deps causal-conv1d mamba-ssm -w "{WHEEL_OUT}"
    !pip install -q --no-deps --force-reinstall "{WHEEL_OUT}"/causal_conv1d-*.whl "{WHEEL_OUT}"/mamba_ssm-*.whl
    open(f"{WHEEL_OUT}/env_tag.txt", "w").write(ENV_TAG)
    !ls -la "{WHEEL_OUT}"
    assert mamba_ok(), "Build xong nhưng kernel mamba vẫn lỗi -> xem log pip phía trên"
%cd {REPO}

In [ ]:
# Kiểm tra kernel CUDA selective_scan của mamba và đường dẫn nnU-Net
check = ("import torch, mamba_ssm; "
         "from mamba_ssm.ops.selective_scan_interface import selective_scan_fn; "
         "u=torch.randn(1,4,16,device='cuda'); d=torch.randn(1,4,16,device='cuda'); A=-torch.rand(4,8,device='cuda'); "
         "B=torch.randn(1,8,16,device='cuda'); C=torch.randn(1,8,16,device='cuda'); "
         "print('torch', torch.__version__, '| mamba_ssm', mamba_ssm.__version__, '| out', selective_scan_fn(u,d,A,B,C).shape)")
subprocess.run([sys.executable, "-c", check], check=True)
!python -c "import nnunetv2, monai; from nnunetv2.paths import nnUNet_raw, nnUNet_results; print('nnUNet_raw =', nnUNet_raw); print('nnUNet_results =', nnUNet_results)"

## 5. Tải pretrained VMamba-Tiny và tạo thư mục dữ liệu
- `data/nnUNet_raw/<dataset>` và `data/nnUNet_preprocessed` là symlink tới `/content/nnunet` (đĩa local, tốc độ nhanh).
- `data/nnUNet_results` là symlink tới `RESULTS_DIR` trên Google Drive. Checkpoint được ghi thẳng lên Drive, nên ngắt runtime cũng không mất.

In [ ]:
os.chdir(REPO)
ckpt = f"{REPO}/data/pretrained/vmamba/vmamba_tiny_e292.pth"
os.makedirs(os.path.dirname(ckpt), exist_ok=True)
import urllib.request
if os.path.isfile(ckpt) and os.path.getsize(ckpt) < 50 * 2**20:   # file hỏng do lần tải trước bị lỗi
    os.remove(ckpt)
if not os.path.isfile(ckpt):
    urllib.request.urlretrieve(PRETRAIN_URL, ckpt + ".part")
    os.rename(ckpt + ".part", ckpt)
assert os.path.getsize(ckpt) > 50 * 2**20, "File pretrained không hợp lệ"
print("pretrained:", ckpt, os.path.getsize(ckpt) // 2**20, "MB")

RAW_DS = f"{TMP}/nnUNet_raw/{DATASET_NAME}"
os.makedirs(RAW_DS, exist_ok=True)
os.makedirs(f"{TMP}/nnUNet_preprocessed", exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(f"{REPO}/data/nnUNet_raw", exist_ok=True)

def force_symlink(link, target):
    if os.path.islink(link):
        if os.readlink(link) == target:
            return
        os.unlink(link)
    elif os.path.isdir(link):          # thư mục thật (vd có sẵn trong repo) -> chép nội dung sang target rồi thay bằng symlink
        subprocess.run(["cp", "-rn", f"{link}/.", target], check=False)
        shutil.rmtree(link)
    os.symlink(target, link)

force_symlink(f"{REPO}/data/nnUNet_raw/{DATASET_NAME}", RAW_DS)
force_symlink(f"{REPO}/data/nnUNet_preprocessed", f"{TMP}/nnUNet_preprocessed")
force_symlink(f"{REPO}/data/nnUNet_results", RESULTS_DIR)
!ls -la {REPO}/data

## 6. Trainer tuỳ chỉnh
Trainer kế thừa trainer gốc của Swin-UMamba (AdamW, cosine schedule, deep supervision, pretrain VMamba-Tiny) và đọc siêu tham số từ biến môi trường do ô cấu hình đặt:

| Tham số | Gốc (~350 epoch) | Ở đây |
|---|---|---|
| `num_epochs` | 1000 (dừng sớm 350) | `EPOCHS` |
| `freeze_encoder_epochs` | 10 | `FREEZE_EPOCHS` |
| `initial_lr` | 1e-4 | `INITIAL_LR` |
| `num_val_iterations_per_epoch` | 50 | `VAL_ITERS` |
| `oversample_foreground_percent` | 0.33 | `OVERSAMPLE_FG` |
| Trọng số CE | không | `CE_CLASS_WEIGHTS` |

Các trainer có sẵn (đặt vào biến `TRAINER`): `nnUNetTrainerSwinUMambaColab`, `nnUNetTrainerSwinUMambaDColab` (Swin-UMamba†), `nnUNetTrainerSwinUMambaScratchColab`, `nnUNetTrainerSwinUMambaDScratchColab` (hai trainer cuối không dùng pretrain).

Trainer không lưu `checkpoint_{epoch}.pth` (khoảng 0.5 GB mỗi file, sẽ làm đầy Drive). Vẫn giữ `checkpoint_latest`, `checkpoint_best` và `checkpoint_final`.

In [ ]:
trainer_src = """
import os
import torch
from nnunetv2.training.nnUNetTrainer.nnUNetTrainer import nnUNetTrainer
from nnunetv2.training.nnUNetTrainer.nnUNetTrainerSwinUMamba import nnUNetTrainerSwinUMamba
from nnunetv2.training.nnUNetTrainer.nnUNetTrainerSwinUMambaD import nnUNetTrainerSwinUMambaD
from nnunetv2.training.nnUNetTrainer.nnUNetTrainerSwinUMambaScratch import nnUNetTrainerSwinUMambaScratch
from nnunetv2.training.nnUNetTrainer.nnUNetTrainerSwinUMambaDScratch import nnUNetTrainerSwinUMambaDScratch
from nnunetv2.training.loss.compound_losses import DC_and_CE_loss


def _custom_setup(self):
    env = os.environ.get
    self.num_epochs = int(env("SUM_EPOCHS", 20))
    self.num_iterations_per_epoch = int(env("SUM_ITERS", 200))
    self.num_val_iterations_per_epoch = int(env("SUM_VAL_ITERS", 20))
    self.save_every = int(env("SUM_SAVE_EVERY", 1))
    self.freeze_encoder_epochs = int(env("SUM_FREEZE_EPOCHS", max(1, round(self.num_epochs * 0.1))))
    self.initial_lr = float(env("SUM_LR", 3e-4))
    self.oversample_foreground_percent = float(env("SUM_OVERSAMPLE", 0.66))
    w = env("SUM_CE_WEIGHTS", "").strip()
    self.ce_class_weights = [float(x) for x in w.split(",")] if w else None


def _build_loss_weighted(self):
    loss = nnUNetTrainer._build_loss(self)
    w = self.ce_class_weights
    if w and not self.label_manager.has_regions:
        assert len(w) == self.label_manager.num_segmentation_heads, f"SUM_CE_WEIGHTS cần {self.label_manager.num_segmentation_heads} giá trị"
        dc_ce = loss.loss if hasattr(loss, "loss") else loss      # bỏ lớp DeepSupervisionWrapper
        assert isinstance(dc_ce, DC_and_CE_loss)
        dc_ce.ce.weight = torch.tensor(w, dtype=torch.float32, device=self.device)
        self.print_to_log_file(f"CE class weights: {w}")
    return loss

"""
for base in ["nnUNetTrainerSwinUMamba", "nnUNetTrainerSwinUMambaD",
             "nnUNetTrainerSwinUMambaScratch", "nnUNetTrainerSwinUMambaDScratch"]:
    trainer_src += f"""
class {base}Colab({base}):
    def __init__(self, plans: dict, configuration: str, fold: int, dataset_json: dict,
                 unpack_dataset: bool = True, device: torch.device = torch.device("cuda")):
        super().__init__(plans, configuration, fold, dataset_json, unpack_dataset, device)
        _custom_setup(self)
        self.print_to_log_file(f"[Colab] epochs={{self.num_epochs}} freeze={{self.freeze_encoder_epochs}} "
                               f"iters={{self.num_iterations_per_epoch}} lr={{self.initial_lr}}")

    def _build_loss(self):
        return _build_loss_weighted(self)

    def on_epoch_end(self):
        nnUNetTrainer.on_epoch_end(self)   # bỏ qua việc lưu checkpoint_{{epoch}}.pth của lớp cha

"""
path = f"{REPO}/swin_umamba/nnunetv2/training/nnUNetTrainer/nnUNetTrainerSwinUMambaColab.py"
open(path, "w").write(trainer_src)
!python -c "from nnunetv2.training.nnUNetTrainer.nnUNetTrainerSwinUMambaColab import *; print('OK')"

## 7. Chuyển BTXRD sang định dạng nnU-Net
- Chỉ dùng nhãn **polygon** (vùng khối u); bỏ qua `rectangle` (bbox).
- Lớp lành/ác lấy từ cột `type` trong `split.csv`. Nếu ảnh không có trong csv thì suy ra từ tên nhãn (`osteosarcoma`, `other mt`, `*sarcoma` → ác tính).
- Mask PNG: 0 = nền, 1 = lành tính, 2 = ác tính.
- `train` + `val` → `imagesTr/labelsTr`; `test` → `imagesTs/labelsTs`. Danh sách train/val được ghi ra `splits_final.json` để fold 0 validate đúng trên tập `val`.

In [ ]:
convert_src = r"""
import os, sys, json, glob, csv
import numpy as np, cv2
from PIL import Image, ImageOps

src, out, meta, mode, max_size, name = sys.argv[1:7]
max_size = int(max_size) if max_size != "None" else None

types = {}
for p in [os.path.join(src, "split.csv"), os.path.join(os.path.dirname(src), "split.csv")]:
    if os.path.isfile(p):
        for r in csv.DictReader(open(p, encoding="utf-8-sig")):
            if r.get("image_id") and r.get("type"):
                types[os.path.splitext(r["image_id"].strip())[0]] = r["type"].strip().lower()
        print("split.csv:", p, "|", len(types), "ảnh có type")
        break

MALIGNANT = {"osteosarcoma", "other mt", "ewing sarcoma", "chondrosarcoma"}
def is_malignant(stem, label):
    if stem in types:
        return types[stem] == "malignant"
    l = label.strip().lower()
    return l in MALIGNANT or "sarcoma" in l

if mode == "binary":
    labels = {"background": 0, "tumor": 1}
    cls = lambda stem, l: 1
else:
    labels = {"background": 0, "benign": 1, "malignant": 2}
    cls = lambda stem, l: 2 if is_malignant(stem, l) else 1

for d in ["imagesTr", "labelsTr", "imagesTs", "labelsTs"]:
    os.makedirs(os.path.join(out, d), exist_ok=True)
os.makedirs(meta, exist_ok=True)

ids = {"train": [], "val": [], "test": []}
info = []
for split in ["train", "val", "test"]:
    sdir = os.path.join(src, split)
    imgs = {os.path.splitext(f)[0]: os.path.join(sdir, "images", f) for f in os.listdir(os.path.join(sdir, "images"))}
    sub = "Ts" if split == "test" else "Tr"
    skipped = 0
    for jf in sorted(glob.glob(os.path.join(sdir, "Annotations", "*.json"))):
        stem = os.path.splitext(os.path.basename(jf))[0]
        ann = json.load(open(jf, encoding="utf-8"))
        polys = [s for s in ann["shapes"] if s.get("shape_type") == "polygon" and len(s["points"]) >= 3]
        if stem not in imgs or not polys:
            skipped += 1
            continue
        img = ImageOps.exif_transpose(Image.open(imgs[stem])).convert("L")
        w, h = img.size
        sx = w / float(ann.get("imageWidth") or w)
        sy = h / float(ann.get("imageHeight") or h)
        mask = np.zeros((h, w), np.uint8)
        for c in (1, 2):   # vẽ ác tính sau để ưu tiên khi chồng lấn
            for s in polys:
                if cls(stem, s["label"]) == c:
                    pts = np.array([[x * sx, y * sy] for x, y in s["points"]], np.float64).round().astype(np.int32)
                    cv2.fillPoly(mask, [pts], c)
        img = np.array(img)
        if max_size and max(h, w) > max_size:
            r = max_size / max(h, w)
            nw, nh = int(round(w * r)), int(round(h * r))
            img = cv2.resize(img, (nw, nh), interpolation=cv2.INTER_AREA)
            mask = cv2.resize(mask, (nw, nh), interpolation=cv2.INTER_NEAREST)
        cv2.imwrite(os.path.join(out, f"images{sub}", f"{stem}_0000.png"), img)
        cv2.imwrite(os.path.join(out, f"labels{sub}", f"{stem}.png"), mask)
        ids[split].append(stem)
        info.append((stem, split, "malignant" if mask.max() == 2 else "benign", "|".join(sorted({s["label"] for s in polys}))))
    print(f"{split}: {len(ids[split])} ảnh (bỏ qua {skipped} ảnh không có polygon/ảnh)")

json.dump({"channel_names": {"0": "X-ray"}, "labels": labels, "numTraining": len(ids["train"]) + len(ids["val"]),
           "file_ending": ".png", "name": name,
           "description": f"BTXRD bone tumor X-ray, polygons -> segmentation masks ({mode})"},
          open(os.path.join(out, "dataset.json"), "w"), indent=2)
json.dump([{"train": sorted(ids["train"]), "val": sorted(ids["val"])}], open(os.path.join(meta, "splits_final.json"), "w"), indent=2)
with open(os.path.join(meta, "cases.csv"), "w", newline="") as f:
    wr = csv.writer(f); wr.writerow(["case", "split", "type", "labels"]); wr.writerows(info)
print("labels:", labels)
"""
open(f"{TMP}/convert_btxrd.py", "w").write(convert_src)
META = f"{TMP}/btxrd_meta_{DATASET_ID}"   # mỗi dataset một thư mục meta

ds_json = f"{RAW_DS}/dataset.json"
n_labels = 3 if LABEL_MODE == "benign_malignant" else 2
if not (os.path.isfile(ds_json) and len(json.load(open(ds_json))["labels"]) == n_labels and os.path.isfile(f"{META}/splits_final.json")):
    !rm -rf {RAW_DS}/* {META}
    !python {TMP}/convert_btxrd.py "{BTXRD_DIR}" "{RAW_DS}" "{META}" {LABEL_MODE} {MAX_SIZE} {DATASET_NAME}
else:
    print("Dữ liệu nnU-Net đã có sẵn, bỏ qua bước chuyển đổi")

!cat {RAW_DS}/dataset.json
for d in ["imagesTr", "labelsTr", "imagesTs", "labelsTs"]:
    print(d, len(glob.glob(f"{RAW_DS}/{d}/*")))

## 8. Plan & preprocess

In [ ]:
os.chdir(REPO)
NP = os.environ["nnUNet_n_proc_DA"]
!nnUNetv2_plan_and_preprocess -d {DATASET_ID} -c 2d -np {NP} --verify_dataset_integrity

# split train/val cố định -> fold 0 của nnU-Net validate đúng trên tập val
!cp {META}/splits_final.json {TMP}/nnUNet_preprocessed/{DATASET_NAME}/splits_final.json
sp = json.load(open(f"{TMP}/nnUNet_preprocessed/{DATASET_NAME}/splits_final.json"))
print("splits_final.json: train", len(sp[0]["train"]), "| val", len(sp[0]["val"]))
if FOLD not in ("0", "all"):
    print("CẢNH BÁO: splits_final.json chỉ có fold 0 -> FOLD khác sẽ bị nnU-Net chia ngẫu nhiên 80/20")

plans_path = f"{TMP}/nnUNet_preprocessed/{DATASET_NAME}/nnUNetPlans.json"
plans = json.load(open(plans_path))
if BATCH_SIZE:
    plans["configurations"]["2d"]["batch_size"] = int(BATCH_SIZE)
    json.dump(plans, open(plans_path, "w"), indent=2)
cfg = plans["configurations"]["2d"]
print("patch_size:", cfg["patch_size"], "| batch_size:", cfg["batch_size"])

## 9. Train + Validation
- Nếu thí nghiệm đã có `checkpoint_final.pth` **và** `validation/summary.json` thì bỏ qua bước train (B0 thuộc trường hợp này).
- Nếu chỉ có `checkpoint_latest.pth` (lần trước bị ngắt), notebook thêm `--c` để train tiếp.
- Cấu hình của thí nghiệm được ghi vào `exp_config.json` trong thư mục kết quả.

In [ ]:
os.chdir(REPO)
OUT_DIR  = f"{REPO}/data/nnUNet_results/{DATASET_NAME}/{TRAINER}__nnUNetPlans__2d"
FOLD_DIR = f"{OUT_DIR}/fold_{FOLD}"
json.dump({"exp": EXP, "desc": EXPERIMENTS[EXP]["desc"], "run_name": RUN_NAME, **CFG,
           "FREEZE_EPOCHS": FREEZE_EPOCHS, "TRAINER": TRAINER, "DATASET": DATASET_NAME},
          open(f"{RESULTS_DIR}/exp_config.json", "w"), indent=2, ensure_ascii=False)

done = os.path.isfile(f"{FOLD_DIR}/checkpoint_final.pth") and os.path.isfile(f"{FOLD_DIR}/validation/summary.json")
if done:
    print(f"[{EXP}] đã train xong trước đó -> bỏ qua bước train")
else:
    resume = "--c" if os.path.isfile(f"{FOLD_DIR}/checkpoint_latest.pth") else ""
    print("resume flag:", resume or "(train mới)")
    print(f"EPOCHS={EPOCHS} FREEZE={FREEZE_EPOCHS} ITERS={ITERS_PER_EPOCH} BATCH={BATCH_SIZE} LR={INITIAL_LR} "
          f"OVERSAMPLE={OVERSAMPLE_FG} CE={CE_CLASS_WEIGHTS or 'none'} MAX_SIZE={MAX_SIZE}")
    !CUDA_VISIBLE_DEVICES=0 nnUNetv2_train {DATASET_ID} 2d {FOLD} -tr {TRAINER} {resume}

In [ ]:
# Kết quả validation & đường cong huấn luyện
from IPython.display import Image as IPImage, display
summ = f"{FOLD_DIR}/validation/summary.json"
CLASS_NAMES = {v: k for k, v in json.load(open(f"{RAW_DS}/dataset.json"))["labels"].items()}
if os.path.isfile(summ):
    s = json.load(open(summ))
    print("Validation mean Dice (foreground):", round(s["foreground_mean"]["Dice"], 4))
    for k, v in s["mean"].items():
        print(f"  class {k} ({CLASS_NAMES.get(int(k), '?')}): Dice={v['Dice']:.4f}  IoU={v['IoU']:.4f}")
else:
    print("Chưa có validation/summary.json (train chưa xong?)")
if os.path.isfile(f"{FOLD_DIR}/progress.png"):
    display(IPImage(f"{FOLD_DIR}/progress.png"))

## 10. Inference trên tập test (`--disable_tta` giống script gốc)

In [ ]:
os.chdir(REPO)
CKPT = next(c for c in ["checkpoint_final.pth", "checkpoint_best.pth", "checkpoint_latest.pth"]
            if os.path.isfile(f"{FOLD_DIR}/{c}"))
PRED_DIR = f"{OUT_DIR}/pred_results_fold{FOLD}"
print("checkpoint:", CKPT)
!CUDA_VISIBLE_DEVICES=0 nnUNetv2_predict -i "{RAW_DS}/imagesTs" -o "{PRED_DIR}" -d {DATASET_ID} -c 2d -tr {TRAINER} -f {FOLD} -chk {CKPT} --disable_tta

## 11. Đánh giá DSC & NSD (script đánh giá gốc cho ảnh PNG 2D)

In [ ]:
os.chdir(REPO)
!python evaluation/endoscopy_DSC_Eval.py --gt_path "{RAW_DS}/labelsTs" --seg_path "{PRED_DIR}" --save_path "{OUT_DIR}/metric_DSC_fold{FOLD}.csv"
!python evaluation/endoscopy_NSD_Eval.py --gt_path "{RAW_DS}/labelsTs" --seg_path "{PRED_DIR}" --save_path "{OUT_DIR}/metric_NSD_fold{FOLD}.csv"

### 11b. Dice theo từng lớp & độ chính xác phân loại lành/ác
Lớp dự đoán của mỗi ảnh là lớp (lành/ác) chiếm nhiều pixel nhất trong mask dự đoán. Nếu mask không có pixel u nào thì ghi là `none`.

In [ ]:
import numpy as np, cv2, pandas as pd
fg = [c for c in sorted(CLASS_NAMES) if c > 0]

def dice(a, b):
    s = a.sum() + b.sum()
    return np.nan if s == 0 else 2.0 * (a & b).sum() / s

rows = []
for n in sorted(f for f in os.listdir(PRED_DIR) if f.endswith(".png")):
    gt = cv2.imread(f"{RAW_DS}/labelsTs/{n}", cv2.IMREAD_UNCHANGED)
    pr = cv2.imread(f"{PRED_DIR}/{n}", cv2.IMREAD_UNCHANGED)
    r = {"case": n[:-4], "Dice_tumor": dice(gt > 0, pr > 0)}
    for c in fg:
        r[f"Dice_{CLASS_NAMES[c]}"] = dice(gt == c, pr == c)
    area = {c: int((pr == c).sum()) for c in fg}
    gt_area = {c: int((gt == c).sum()) for c in fg}
    r["gt_class"] = CLASS_NAMES[max(gt_area, key=gt_area.get)]
    r["pred_class"] = CLASS_NAMES[max(area, key=area.get)] if sum(area.values()) else "none"
    rows.append(r)
df = pd.DataFrame(rows)
df.to_csv(f"{OUT_DIR}/metric_class_fold{FOLD}.csv", index=False)
pd.set_option("display.width", 200)
print(df.round(4).to_string(index=False))
print("\nMean Dice (bỏ qua ảnh không có lớp đó ở cả GT và dự đoán):")
print(df.filter(like="Dice_").mean().round(4).to_string())
if len(fg) > 1:
    acc = (df.gt_class == df.pred_class).mean()
    print(f"\nĐộ chính xác phân loại lành/ác (theo ảnh): {acc:.4f}  ({(df.gt_class == df.pred_class).sum()}/{len(df)})")
    print(pd.crosstab(df.gt_class, df.pred_class, rownames=["GT"], colnames=["Pred"]))

## 12. Trực quan hoá vài kết quả

In [ ]:
import random
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
from matplotlib.patches import Patch
%matplotlib inline
CMAP = ListedColormap(["black", "lime", "red"])   # 1 = lành tính (xanh), 2 = ác tính (đỏ)

names = sorted(n for n in os.listdir(PRED_DIR) if n.endswith(".png"))
random.Random(0).shuffle(names)
k = min(4, len(names))
fig, ax = plt.subplots(k, 3, figsize=(12, 4 * k), squeeze=False)
for i, n in enumerate(names[:k]):
    img = cv2.imread(f"{RAW_DS}/imagesTs/{n[:-4]}_0000.png", cv2.IMREAD_GRAYSCALE)
    gt  = cv2.imread(f"{RAW_DS}/labelsTs/{n}", cv2.IMREAD_UNCHANGED)
    pr  = cv2.imread(f"{PRED_DIR}/{n}", cv2.IMREAD_UNCHANGED)
    ax[i, 0].imshow(img, cmap="gray"); ax[i, 0].set_title(n)
    ax[i, 1].imshow(img, cmap="gray"); ax[i, 1].imshow(np.ma.masked_equal(gt, 0), alpha=0.5, cmap=CMAP, vmin=0, vmax=2, interpolation="nearest"); ax[i, 1].set_title("Ground truth")
    ax[i, 2].imshow(img, cmap="gray"); ax[i, 2].imshow(np.ma.masked_equal(pr, 0), alpha=0.5, cmap=CMAP, vmin=0, vmax=2, interpolation="nearest"); ax[i, 2].set_title("Prediction")
    for a in ax[i]: a.axis("off")
fig.legend(handles=[Patch(color="lime", label="lành tính (1)"), Patch(color="red", label="ác tính (2)")], loc="upper center", ncol=2)
plt.tight_layout(rect=(0, 0, 1, 0.97)); plt.show()

### 12b. Failure cases
Hai nhóm lỗi chính trên tập test:
- **Bỏ sót**: mặt nạ dự đoán không có pixel u nào. Notebook so sánh diện tích u (tỉ lệ % ảnh) của nhóm bỏ sót và nhóm phát hiện được, rồi vẽ các ca bỏ sót có u nhỏ nhất.
- **Lành → ác**: ảnh u lành tính bị gán thành ác tính.

Hình được lưu vào `failure_cases.png` trong thư mục kết quả của thí nghiệm.

In [ ]:
meta_csv = f"{TMP}/btxrd_meta_{DATASET_ID}/cases.csv"
tumor_types = pd.read_csv(meta_csv).set_index("case")["labels"] if os.path.isfile(meta_csv) else pd.Series(dtype=str)

df["gt_area_pct"] = [100.0 * (cv2.imread(f"{RAW_DS}/labelsTs/{c}.png", cv2.IMREAD_UNCHANGED) > 0).mean() for c in df.case]
df["tumor_type"] = [tumor_types.get(c, "?") for c in df.case]
miss = df[df.pred_class == "none"]
det  = df[df.pred_class != "none"]
b2m  = df[(df.gt_class == "benign") & (df.pred_class == "malignant")]
m2b  = df[(df.gt_class == "malignant") & (df.pred_class == "benign")]
print(f"Bỏ sót: {len(miss)} ảnh | lành->ác: {len(b2m)} | ác->lành: {len(m2b)}")
if len(miss) and len(det):
    print(f"Diện tích u (% ảnh), trung vị: nhóm bỏ sót {miss.gt_area_pct.median():.2f}% "
          f"| nhóm phát hiện được {det.gt_area_pct.median():.2f}%")
cols = ["case", "gt_class", "pred_class", "Dice_tumor", "gt_area_pct", "tumor_type"]
print("\nCác ca bỏ sót:\n", miss[cols].sort_values("gt_area_pct").round(3).to_string(index=False))
print("\nCác ca lành -> ác:\n", b2m[cols].sort_values("Dice_tumor", ascending=False).round(3).to_string(index=False))
df.to_csv(f"{OUT_DIR}/metric_class_fold{FOLD}.csv", index=False)

cases = [(r, "bỏ sót") for _, r in miss.sort_values("gt_area_pct").head(3).iterrows()] + \
        [(r, "lành → ác") for _, r in b2m.sort_values("Dice_tumor", ascending=False).head(3).iterrows()]
if cases:
    fig, ax = plt.subplots(len(cases), 3, figsize=(11, 3.4 * len(cases)), squeeze=False)
    for i, (r, kind) in enumerate(cases):
        img = cv2.imread(f"{RAW_DS}/imagesTs/{r.case}_0000.png", cv2.IMREAD_GRAYSCALE)
        gt  = cv2.imread(f"{RAW_DS}/labelsTs/{r.case}.png", cv2.IMREAD_UNCHANGED)
        pr  = cv2.imread(f"{PRED_DIR}/{r.case}.png", cv2.IMREAD_UNCHANGED)
        ax[i, 0].imshow(img, cmap="gray"); ax[i, 0].set_title(f"{r.case} ({kind})\n{r.tumor_type}", fontsize=9)
        ax[i, 1].imshow(img, cmap="gray"); ax[i, 1].imshow(np.ma.masked_equal(gt, 0), alpha=0.5, cmap=CMAP, vmin=0, vmax=2, interpolation="nearest")
        ax[i, 1].set_title(f"Ground truth (u chiếm {r.gt_area_pct:.1f}% ảnh)", fontsize=9)
        ax[i, 2].imshow(img, cmap="gray"); ax[i, 2].imshow(np.ma.masked_equal(pr, 0), alpha=0.5, cmap=CMAP, vmin=0, vmax=2, interpolation="nearest")
        ax[i, 2].set_title(f"Dự đoán (Dice khối u {r.Dice_tumor:.2f})", fontsize=9)
        for a in ax[i]: a.axis("off")
    fig.legend(handles=[Patch(color="lime", label="lành tính (1)"), Patch(color="red", label="ác tính (2)")], loc="upper center", ncol=2)
    plt.tight_layout(rect=(0, 0, 1, 0.97))
    plt.savefig(f"{RESULTS_DIR}/failure_cases.png", dpi=150, bbox_inches="tight")
    plt.show()
    print("Đã lưu:", f"{RESULTS_DIR}/failure_cases.png")

## 13. Kết quả đã lưu
Toàn bộ checkpoint, log, `progress.png`, kết quả validation, dự đoán trên tập test và các file CSV metric nằm trong `RESULTS_DIR` trên Google Drive (`MyDrive/Swin-UMamba-BTXRD/results/<run_name>/`).

In [ ]:
!du -sh "{RESULTS_DIR}"
!ls -la "{FOLD_DIR}"

## 14. Độ phức tạp, tốc độ và tóm tắt thí nghiệm
- **Số tham số, GMAC**: lấy từ `torchinfo.summary` mà trainer Swin-UMamba gọi khi dựng mạng (cùng cách với log huấn luyện), đầu vào 1 patch, không deep supervision. Lưu ý: torchinfo không đếm phép tính trong kernel CUDA selective-scan của Mamba, nên GMAC là cận dưới.
- **Thời gian/epoch**: trung vị các dòng `Epoch time` trong log huấn luyện.
- **Thời gian suy luận/ảnh**: đo end-to-end cho từng ảnh test bằng `nnUNetPredictor.predict_single_npy_array` (tiền xử lý + cửa sổ trượt + đưa về kích thước gốc, không TTA, không tính đọc/ghi đĩa), bỏ 3 ảnh đầu để làm nóng GPU. Thêm thời gian forward của 1 patch.
- Mọi chỉ số được ghi vào `summary_run.json` để mục 15 gộp lại.

In [ ]:
import io, re, time, contextlib
import torch
from nnunetv2.inference.predict_from_raw_data import nnUNetPredictor
from nnunetv2.imageio.natural_image_reader_writer import NaturalImage2DIO

os.chdir(REPO)   # trainer nạp pretrained theo đường dẫn tương đối data/pretrained/...
predictor = nnUNetPredictor(tile_step_size=0.5, use_gaussian=True, use_mirroring=False,
                            perform_everything_on_device=True, device=torch.device("cuda"),
                            verbose=False, verbose_preprocessing=False, allow_tqdm=False)
buf = io.StringIO()
with contextlib.redirect_stdout(buf):   # build_network_architecture in bảng torchinfo -> bắt lại để đọc số liệu
    predictor.initialize_from_trained_model_folder(OUT_DIR, use_folds=(FOLD,), checkpoint_name=CKPT)
txt = buf.getvalue()
m_par = re.search(r"Total params:\s*([\d,]+)", txt)
m_mac = re.search(r"Total mult-adds \((G|M|T)\):\s*([\d.]+)", txt)
params_M = (int(m_par.group(1).replace(",", "")) if m_par else sum(p.numel() for p in predictor.network.parameters())) / 1e6
gmac = (float(m_mac.group(2)) * {"M": 1e-3, "G": 1, "T": 1e3}[m_mac.group(1)]) if m_mac else float("nan")
patch = list(predictor.configuration_manager.patch_size)
print(f"Params: {params_M:.2f} M | GMAC (1 patch {patch}): {gmac:.1f}")

# --- thời gian suy luận end-to-end / ảnh
reader = NaturalImage2DIO()
files = sorted(glob.glob(f"{RAW_DS}/imagesTs/*_0000.png"))
times = []
for i, f in enumerate(files):
    img, props = reader.read_images([f])
    torch.cuda.synchronize(); t0 = time.perf_counter()
    _ = predictor.predict_single_npy_array(img, props, None, None, False)
    torch.cuda.synchronize(); times.append(time.perf_counter() - t0)
    if i == 0:   # giữ trọng số trên GPU để không tính thời gian chép state_dict CPU->GPU ở mỗi ảnh
        predictor.list_of_parameters = [{k: v.to("cuda") for k, v in p.items()} for p in predictor.list_of_parameters]
infer_ms = 1000 * float(np.mean(times[3:]))
print(f"Suy luận end-to-end: {infer_ms:.1f} ms/ảnh (trung bình {len(times) - 3} ảnh)")

# --- forward 1 patch
net = predictor.network.to("cuda").eval()
x = torch.randn(1, 1, *patch, device="cuda")
with torch.no_grad(), torch.autocast("cuda"):
    for _ in range(5): net(x)
    torch.cuda.synchronize(); t0 = time.perf_counter()
    for _ in range(20): net(x)
    torch.cuda.synchronize()
patch_ms = 1000 * (time.perf_counter() - t0) / 20
print(f"Forward 1 patch: {patch_ms:.1f} ms")

# --- thời gian huấn luyện từ log
ep_times = []
for lf in sorted(glob.glob(f"{FOLD_DIR}/training_log_*.txt")):
    ep_times += [float(t) for t in re.findall(r"Epoch time: ([\d.]+) s", open(lf).read())]
epoch_s = float(np.median(ep_times)) if ep_times else float("nan")
print(f"Thời gian/epoch (trung vị {len(ep_times)} epoch trong log): {epoch_s:.0f} s")

# --- gộp chỉ số
val = json.load(open(f"{FOLD_DIR}/validation/summary.json"))
dsc = pd.read_csv(f"{OUT_DIR}/metric_DSC_fold{FOLD}.csv")["DSC"].mean()
nsd = pd.read_csv(f"{OUT_DIR}/metric_NSD_fold{FOLD}.csv")["NSD"].mean()
cm = pd.crosstab(df.gt_class, df.pred_class)
get = lambda g, p: int(cm.loc[g, p]) if (g in cm.index and p in cm.columns) else 0
n_mal, n_pred_mal = int((df.gt_class == "malignant").sum()), int((df.pred_class == "malignant").sum())
summary = {
    "exp": EXP, "desc": EXPERIMENTS[EXP]["desc"], "run_name": RUN_NAME, **CFG,
    "val_dice": round(val["foreground_mean"]["Dice"], 4),
    "test_DSC": round(dsc, 4), "test_NSD": round(nsd, 4),
    "dice_tumor": round(df.Dice_tumor.mean(), 4),
    "dice_benign": round(df.Dice_benign.mean(), 4), "dice_malignant": round(df.Dice_malignant.mean(), 4),
    "accuracy": round(float((df.gt_class == df.pred_class).mean()), 4),
    "recall_malignant": round(get("malignant", "malignant") / max(n_mal, 1), 4),
    "precision_malignant": round(get("malignant", "malignant") / max(n_pred_mal, 1), 4),
    "missed": int((df.pred_class == "none").sum()),
    "benign_as_malignant": get("benign", "malignant"), "malignant_as_benign": get("malignant", "benign"),
    "params_M": round(params_M, 2), "GMAC": round(gmac, 1), "patch": patch,
    "epoch_time_s": round(epoch_s, 1), "train_time_min": round(sum(ep_times) / 60, 1),
    "infer_ms_per_image": round(infer_ms, 1), "forward_ms_per_patch": round(patch_ms, 1),
    "gpu": torch.cuda.get_device_name(0),
}
json.dump(summary, open(f"{RESULTS_DIR}/summary_run.json", "w"), indent=2, ensure_ascii=False)
print(json.dumps(summary, indent=2, ensure_ascii=False))

## 15. So sánh các thí nghiệm
Đọc mọi `results/*/summary_run.json` đã có trên Drive (không cần chạy lại các thí nghiệm khác), in bảng so sánh, lưu `results/comparison.csv` và vẽ biểu đồ đánh đổi `results/tradeoff.png`.

In [ ]:
import matplotlib.pyplot as plt
rows = [json.load(open(f)) for f in sorted(glob.glob(f"{RESULTS_ROOT}/*/summary_run.json"))]
cmp = pd.DataFrame(rows).sort_values("exp").reset_index(drop=True)
cmp.to_csv(f"{RESULTS_ROOT}/comparison.csv", index=False)

show = {"exp": "Thí nghiệm", "desc": "Thay đổi", "dice_tumor": "Dice khối u", "test_DSC": "DSC", "test_NSD": "NSD",
        "accuracy": "Accuracy", "recall_malignant": "Recall ác", "precision_malignant": "Precision ác",
        "missed": "Bỏ sót", "benign_as_malignant": "Lành→ác", "val_dice": "Val Dice",
        "params_M": "Params (M)", "GMAC": "GMAC", "epoch_time_s": "s/epoch", "infer_ms_per_image": "ms/ảnh"}
tbl = cmp[[c for c in show if c in cmp]].rename(columns=show)
pd.set_option("display.width", 250)
print(tbl.to_string(index=False))
print("\nBảng Markdown (dán vào báo cáo/README):\n")
try:
    print(tbl.to_markdown(index=False))
except ImportError:          # thiếu gói tabulate
    print(tbl.to_csv(index=False))

# Đánh đổi: độ chính xác vs chi phí suy luận / huấn luyện
fig, ax = plt.subplots(1, 2, figsize=(12, 4.5))
for a, xcol, xlabel in [(ax[0], "infer_ms_per_image", "Thời gian suy luận (ms/ảnh)"),
                        (ax[1], "epoch_time_s", "Thời gian huấn luyện (s/epoch)")]:
    a.scatter(cmp[xcol], cmp["dice_tumor"], s=60, label="Dice khối u")
    a.scatter(cmp[xcol], cmp["accuracy"], s=60, marker="s", label="Accuracy lành/ác")
    for _, r in cmp.iterrows():
        a.annotate(r.exp, (r[xcol], r.dice_tumor), textcoords="offset points", xytext=(5, 5))
    a.set_xlabel(xlabel); a.set_ylabel("Giá trị"); a.grid(alpha=0.3)
ax[0].legend()
plt.tight_layout(); plt.savefig(f"{RESULTS_ROOT}/tradeoff.png", dpi=150, bbox_inches="tight"); plt.show()
print("Đã lưu:", f"{RESULTS_ROOT}/comparison.csv", "|", f"{RESULTS_ROOT}/tradeoff.png")